# 01 · Explore one part

Goal: look at the data before building anything, and write down what "normal" looks like.

By default this notebook reads the **fake sample data**. To look at the real NIST data, change `PART_PATH` and `LAYER_MM` in the first code cell (after `python scripts/download_data.py --parts 1`).

Write what you find into `docs/data_notes.md`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))

from src.features import assign_tracks
from src.labels import add_geometry, add_xct_surface, xct_threshold
from src.load import load_part

# --- sample data (fake) ---
PART_PATH = REPO / "data/sample/part1"
LAYER_MM = 0.08
# --- real NIST data: uncomment these two lines ---
# PART_PATH = REPO / "data/raw/part1"
# LAYER_MM = 0.02

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "font.size": 9})
BLUES = "Blues"


## 1. Load the part

One row is one melt pool camera frame. `load_part` adds the layer number (from the file name) and the height `z`.

In [ ]:
points = load_part(PART_PATH, layer_thickness_mm=LAYER_MM)
points = add_geometry(points, LAYER_MM)        # adds solid_below and is_edge
points = add_xct_surface(points, "xct_m3", LAYER_MM)   # adds xct_surface (same rule as the pipeline)
print(f"{len(points):,} rows, {points['layer'].nunique()} layers, {points.shape[1]} columns")
points.head()

In [ ]:
points[["time_us", "x", "y", "cmd_power", "power", "speed",
        "mp_length_100", "mp_width_100", "mp_area_100", "xct_m3"]].describe().T

## 2. How much data per layer?

Layers with an overhang or a hole have a different area, so a different number of frames.

In [ ]:
per_layer = points.groupby("layer").size()
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot(per_layer.index, per_layer.values, color="#2a78d6", lw=2)
ax.set_xlabel("layer"); ax.set_ylabel("frames"); ax.set_title("Frames per layer", loc="left");

## 3. The signal over time

Melt pool area for the first frames of one layer. Grey lines mark where a new scan track starts (the laser was off in between).

In [ ]:
LAYER = int(points["layer"].median())
lay = points[points["layer"] == LAYER].reset_index(drop=True)
track = assign_tracks(lay)
n = min(600, len(lay))
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.plot(np.arange(n), lay["mp_area_100"][:n], color="#0b0b0b", lw=1)
for start in np.flatnonzero(np.diff(track[:n]) > 0) + 1:
    ax.axvline(start, color="#c3c2b7", lw=0.8)
ax.set_xlabel("frame number in the layer"); ax.set_ylabel("melt pool area [mm²]")
ax.set_title(f"Layer {LAYER}: melt pool area, frame by frame", loc="left");

## 4. The signal on the x/y plane

Each dot is one frame at its laser position. Look for regions that are always brighter or darker.

In [ ]:
layers = np.unique(np.quantile(points["layer"], [0.15, 0.5, 0.9]).astype(int))
vmin, vmax = points["mp_area_100"].quantile([0.01, 0.99])
fig, axes = plt.subplots(1, len(layers), figsize=(4.2 * len(layers), 3.2), squeeze=False)
for ax, l in zip(axes[0], layers):
    d = points[points["layer"] == l]
    sc = ax.scatter(d["x"], d["y"], c=d["mp_area_100"], s=4, cmap=BLUES, vmin=vmin, vmax=vmax)
    ax.set_aspect("equal"); ax.grid(False); ax.set_title(f"layer {l}", loc="left"); ax.set_xlabel("x [mm]")
axes[0][0].set_ylabel("y [mm]")
fig.colorbar(sc, ax=axes[0].tolist(), shrink=0.8, label="melt pool area [mm²]");

## 5. Laser settings

Different commanded powers (for example contour and infill) give different melt pools. "Normal" has to be judged per setting.

In [ ]:
points["setting_W"] = (points["cmd_power"] / 5).round() * 5
print(points.groupby("setting_W")["mp_area_100"].agg(["size", "mean", "std"]))
fig, ax = plt.subplots(figsize=(7, 2.8))
for (power, d), color in zip(points.groupby("setting_W"), ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]):
    ax.hist(d["mp_area_100"], bins=80, density=True, histtype="step", lw=2, color=color, label=f"{power:.0f} W")
ax.set_xlabel("melt pool area [mm²]"); ax.set_ylabel("density"); ax.legend(frameon=False)
ax.set_title("Melt pool area by commanded laser power", loc="left");

## 6. Geometry: how much solid metal is below?

`solid_below` is worked out from the data: an x/y cell is "metal" in a layer if the laser was there. Small values mean the laser is melting over powder (overhang).

In [ ]:
top = int(points["layer"].quantile(0.8))
d = points[points["layer"] == top]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
sc = axes[0].scatter(d["x"], d["y"], c=d["solid_below"], s=4, cmap=BLUES)
axes[0].set_aspect("equal"); axes[0].grid(False)
axes[0].set_title(f"layer {top}: solid metal below [mm]", loc="left")
axes[0].set_xlabel("x [mm]"); axes[0].set_ylabel("y [mm]")
fig.colorbar(sc, ax=axes[0], shrink=0.8)

bins = pd.cut(points["solid_below"], [-0.01, 0.1, 0.25, 0.5, 0.75, 1.0])
by = points.groupby(bins, observed=True)["mp_area_100"].agg(["mean", "std", "size"])
axes[1].errorbar(range(len(by)), by["mean"], yerr=by["std"], fmt="o-", color="#2a78d6", capsize=3)
axes[1].set_xticks(range(len(by))); axes[1].set_xticklabels([str(i) for i in by.index], rotation=20)
axes[1].set_xlabel("solid metal below [mm]"); axes[1].set_ylabel("melt pool area [mm²]")
axes[1].set_title("Melt pool area vs. metal below (mean ± std)", loc="left")
by

## 7. X-ray CT values

**Important check on the real data.** Solid metal should give one narrow peak. Pores are the tail on one side. Decide:

- are pores dark (low values) or bright (high values)?
- is the automatic threshold (dashed line) sensible?

Points near any surface of the part are left out (`xct_surface`: sides, top faces, faces over powder, the lowest layers of the XCT scan), because XCT there mixes metal and air. Points without an XCT value are left out too. `scripts/check_xct.py` does the same check for whole parts.

In [ ]:
inner = points[~points["xct_surface"]]        # also drops points without an XCT value
thr = xct_threshold(inner["xct_m3"].to_numpy(), pore_is="low", k=6.0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(inner["xct_m3"], bins=120, color="#2a78d6")
axes[0].set_yscale("log"); axes[0].axvline(thr, color="#d03b3b", ls="--")
axes[0].set_xlabel("XCT value (3x3x3 mean)"); axes[0].set_ylabel("frames (log scale)")
axes[0].set_title(f"XCT values away from the surface; auto threshold = {thr:.3f}", loc="left")

pores = inner[inner["xct_m3"] < thr]
axes[1].scatter(points["x"][::20], points["z"][::20], s=1, color="#e4e3df")
axes[1].scatter(pores["x"], pores["z"], s=6, color="#d03b3b")
axes[1].set_xlabel("x [mm]"); axes[1].set_ylabel("z [mm]"); axes[1].grid(False)
axes[1].set_title(f"Where the pore points are, side view ({len(pores):,} points)", loc="left")
print(f"{len(inner):,} of {len(points):,} points are away from the surface; "
      f"{len(pores) / len(inner):.3%} of them are below the threshold")

## 8. Layer camera

Brightness of the layer image after the laser scan, at each laser position.

In [ ]:
d = points[points["layer"] == LAYER]
fig, ax = plt.subplots(figsize=(5.5, 3.2))
sc = ax.scatter(d["x"], d["y"], c=d["lwi_scan_a_m3"], s=4, cmap="Greys_r")
ax.set_aspect("equal"); ax.grid(False); ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
ax.set_title(f"layer {LAYER}: layer camera after scan (light A, 3x3 mean)", loc="left")
fig.colorbar(sc, ax=ax, shrink=0.8);

## 9. Write it down

Fill in `docs/data_notes.md`:

- What does a normal melt pool look like, per laser setting?
- How different is the overhang region?
- What happens at the start of each track?
- Are pores dark or bright in XCT, and which threshold will you use?
- Anything odd?